# AI Wordle Solver

A cleaned, single-notebook AI Wordle project with five preserved solver strategies, a Gradio interface, and reproducible benchmarking.

## Imports

In [1]:
import math
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Callable, Dict, Iterable, List, Sequence, Tuple

import gradio as gr

try:
    import pandas as pd
except ImportError:  # The benchmark still works without pandas.
    pd = None


## Dataset

In [2]:
WORD_BANK = """ABOUT ABOVE ABUSE ACTOR ACUTE ADMIT ADOPT ADULT AFTER AGAIN AGENT AGREE AHEAD ALARM ALBUM ALERT ALIKE ALIVE ALLOW ALONE ALONG ALTER AMONG ANGER ANGLE APART APPLE APPLY ARENA ARGUE ARISE ARMED ASIDE ASSET AUDIO AUDIT AVOID AWAKE AWARD AWARE BADGE BASIC BASIN BEACH BEARD BEAST BEGIN BEING BELOW BENCH BIRTH BLACK BLADE BLAME BLEND BLIND BLOCK BLOOD BOARD BOOST BOOTH BOUND BRAIN BRAND BREAD BREAK BREED BRIEF BRING BROAD BROKE BROWN BUILD BUILT BUYER CABLE CARRY CATCH CAUSE CHAIN CHAIR CHART CHASE CHEAP CHECK CHEST CHIEF CHILD CHINA CHOSE CIVIL CLAIM CLASS CLEAN CLEAR CLERK CLICK CLOCK CLOSE COACH COAST COULD COUNT COURT COVER CRAFT CRASH CREAM CRIME CROSS CROWD CROWN CURVE CYCLE DAILY DANCE DEALT DEATH DEBUT DELAY DEPTH DOING DOUBT DOZEN DRAFT DRAMA DRAWN DREAM DRESS DRILL DRINK DRIVE DROVE DYING EAGER EARLY EARTH EIGHT ELBOW ELDER ELECT ELITE EMPTY ENEMY ENJOY ENTER ENTRY EQUAL ERROR EVENT EVERY EXACT EXIST EXTRA FAITH FALSE FAULT FIBER FIELD FIFTH FIFTY FIGHT FINAL FIRST FIXED FLASH FLEET FLOOR FLUID FOCUS FORCE FORTH FORTY FORUM FOUND FRAME FRANK FRAUD FRESH FRONT FRUIT FULLY FUNNY GIANT GIVEN GLASS GLOBE GOING GRACE GRADE GRAND GRANT GRASS GREAT GREEN GROSS GROUP GROWN GUARD GUESS GUEST GUIDE HAPPY HEART HEAVY HENCE HORSE HOTEL HOUSE HUMAN IDEAL IMAGE INDEX INNER INPUT ISSUE JOINT JUDGE KNOWN LABEL LARGE LASER LATER LAUGH LAYER LEARN LEAST LEAVE LEGAL LEVEL LIGHT LIMIT LOCAL LOGIC LOOSE LOWER LUCKY LUNCH MAJOR MAKER MARCH MATCH MAYBE MAYOR MEANT MEDIA METAL MIGHT MINOR MINUS MODEL MONEY MONTH MORAL MOTOR MOUNT MOUSE MOUTH MOVIE MUSIC NEEDS NEVER NIGHT NOISE NORTH NOTED NOVEL NURSE OCCUR OCEAN OFFER OFTEN ORDER OTHER OUGHT PAINT PANEL PAPER PARTY PEACE PHASE PHONE PHOTO PIECE PILOT PITCH PLACE PLAIN PLANE PLANT PLATE POINT POUND POWER PRESS PRICE PRIDE PRIME PRINT PRIOR PRIZE PROOF PROUD PROVE QUEEN QUICK QUIET QUITE RADIO RAISE RANGE RAPID RATIO REACH READY REFER RIGHT RIVAL RIVER ROUGH ROUND ROUTE ROYAL RURAL SCALE SCENE SCOPE SCORE SENSE SERVE SEVEN SHALL SHAPE SHARE SHARP SHEET SHELF SHELL SHIFT SHIRT SHOCK SHOOT SHORT SHOWN SIGHT SINCE SIXTH SIXTY SKILL SLEEP SLIDE SMALL SMART SMILE SMITH SMOKE SOLID SOLVE SORRY SOUND SOUTH SPACE SPARE SPEAK SPEED SPEND SPENT SPLIT SPOKE SPORT STAFF STAGE STAKE STAND START STATE STEAM STEEL STICK STILL STOCK STONE STOOD STORE STORM STORY STRIP STUCK STUDY STUFF STYLE SUGAR SUITE SUPER SWEET TABLE TAKEN TASTE TEACH TEETH THANK THEIR THEME THERE THICK THING THINK THIRD THOSE THREE THROW TIGHT TIMES TIRED TITLE TODAY TOPIC TOTAL TOUCH TOUGH TOWER TRACK TRADE TRAIN TREAT TREND TRIAL TRIED TRUCK TRULY TRUST TRUTH TWICE UNDER UNION UNITY UNTIL UPPER UPSET URBAN USAGE USUAL VALID VALUE VIDEO VIRUS VISIT VITAL VOICE WASTE WATCH WATER WHEEL WHERE WHICH WHILE WHITE WHOLE WHOSE WOMAN WORLD WORRY WOULD WRITE WRONG WROTE YIELD YOUNG YOUTH"""


def load_words(raw_words: str = WORD_BANK) -> List[str]:
    """Return a sorted, unique list of valid five-letter uppercase words.

    The original notebook downloaded `wordlist.txt` at runtime. For a portfolio
    notebook, a small embedded Wordle-style vocabulary is more reliable: the
    project runs offline, in GitHub previews, and in hosted notebooks without any
    extra files.
    """
    words = {word.strip().upper() for word in raw_words.split()}
    valid_words = sorted(word for word in words if len(word) == 5 and word.isalpha())
    if not valid_words:
        raise ValueError("The word bank is empty. Add five-letter words to WORD_BANK.")
    return valid_words


WORDS = load_words()
WORD_SET = set(WORDS)
MAX_ATTEMPTS = 6

print(f"Loaded {len(WORDS)} five-letter words.")


Loaded 469 five-letter words.


## Core Wordle Logic

In [3]:
Feedback = str
History = List[Tuple[str, Feedback]]


def normalize_word(word: str) -> str:
    """Normalize user input into the uppercase format used by the solvers."""
    return word.strip().upper()


def get_feedback(guess: str, secret: str) -> Feedback:
    """Return Wordle feedback for a guess.

    Feedback uses `G` for green, `Y` for yellow, and `B` for black/gray. The
    two-pass implementation correctly handles repeated letters by consuming the
    secret letter counts after exact matches first.
    """
    guess = normalize_word(guess)
    secret = normalize_word(secret)
    feedback = [""] * 5
    remaining = Counter(secret)

    for i, letter in enumerate(guess):
        if letter == secret[i]:
            feedback[i] = "G"
            remaining[letter] -= 1

    for i, letter in enumerate(guess):
        if feedback[i]:
            continue
        if remaining[letter] > 0:
            feedback[i] = "Y"
            remaining[letter] -= 1
        else:
            feedback[i] = "B"

    return "".join(feedback)


def filter_words(candidates: Sequence[str], guess: str, feedback: Feedback) -> List[str]:
    """Keep only candidates that would produce the same feedback pattern."""
    return [word for word in candidates if get_feedback(guess, word) == feedback]


def is_valid_guess(word: str) -> bool:
    """Return True when a guess is exactly five letters and in the dataset."""
    return normalize_word(word) in WORD_SET


def render_grid(history: History) -> str:
    """Render a Wordle board as lightweight HTML for Gradio."""
    colors = {"G": "#6aaa64", "Y": "#c9b458", "B": "#787c7e"}
    tile_base = (
        "width:50px;height:50px;border:2px solid #999;border-radius:6px;"
        "display:flex;align-items:center;justify-content:center;font-weight:700;"
        "font-size:24px;color:white;margin:4px;box-shadow:0 2px 5px rgba(0,0,0,.12);"
    )
    empty_tile = (
        "width:50px;height:50px;border:2px solid #ccc;border-radius:6px;"
        "margin:4px;background:#f5f5f5;"
    )

    rows = ["<div style='font-family:Inter,Arial,sans-serif;display:inline-block;'>"]
    for guess, feedback in history:
        rows.append("<div style='display:flex;'>")
        for letter, mark in zip(guess, feedback):
            rows.append(f"<div style='{tile_base}background:{colors[mark]};'>{letter}</div>")
        rows.append("</div>")

    for _ in range(max(0, MAX_ATTEMPTS - len(history))):
        rows.append("<div style='display:flex;'>")
        rows.extend(f"<div style='{empty_tile}'></div>" for _ in range(5))
        rows.append("</div>")

    rows.append("</div>")
    return "".join(rows)


## AI Algorithms

In [4]:
def random_guess(candidates: Sequence[str], history: History | None = None) -> str:
    """Easy baseline: guess any word from the full dataset."""
    return random.choice(WORDS)


def random_candidate_guess(candidates: Sequence[str], history: History | None = None) -> str:
    """Guess randomly from the currently valid candidate set."""
    return random.choice(list(candidates) or WORDS)


def positional_frequency_guess(candidates: Sequence[str], history: History | None = None) -> str:
    """Score candidates by how common each letter is in its exact position."""
    if not candidates:
        return random_guess(candidates, history)

    position_counts = [Counter(word[i] for word in candidates) for i in range(5)]
    return max(candidates, key=lambda word: sum(position_counts[i][letter] for i, letter in enumerate(word)))


def set_frequency_guess(candidates: Sequence[str], history: History | None = None) -> str:
    """Score candidates by unique-letter frequency in the remaining set.

    This preserves the original notebook's constraint-based/frequency strategy,
    previously named `ai_guess_cbp_fixed`.
    """
    if not candidates:
        return random_guess(candidates, history)

    frequencies = Counter(letter for word in candidates for letter in set(word))
    return max(candidates, key=lambda word: sum(frequencies[letter] for letter in set(word)))


def fast_entropy_heuristic(candidates: Sequence[str], guess: str, sample_limit: int = 200) -> float:
    """Estimate information gain for a guess using Shannon entropy.

    Large candidate pools are sampled to keep interactive play responsive. This
    matches the optimized entropy behavior from the original notebook.
    """
    if not candidates:
        return 0.0

    sampled = random.sample(list(candidates), min(sample_limit, len(candidates))) if len(candidates) > sample_limit else candidates
    pattern_counts = Counter(get_feedback(guess, secret) for secret in sampled)
    total = len(sampled)
    return -sum((count / total) * math.log2(count / total) for count in pattern_counts.values())


def minimax_heuristic(candidates: Sequence[str], guess: str) -> float:
    """Prefer guesses that minimize the largest remaining feedback partition."""
    if not candidates:
        return 0.0
    pattern_counts = Counter(get_feedback(guess, secret) for secret in candidates)
    return -max(pattern_counts.values()) / len(candidates)


def expected_size_heuristic(candidates: Sequence[str], guess: str) -> float:
    """Prefer guesses with the smallest expected remaining candidate count."""
    if not candidates:
        return 0.0
    pattern_counts = Counter(get_feedback(guess, secret) for secret in candidates)
    return -sum(count * count for count in pattern_counts.values()) / len(candidates)


def positional_entropy_heuristic(candidates: Sequence[str], guess: str) -> float:
    """Approximate entropy from per-position letter distributions."""
    if not candidates:
        return 0.0

    total = len(candidates)
    score = 0.0
    for pos in range(5):
        counts = Counter(word[pos] for word in candidates)
        entropy = -sum((count / total) * math.log2(count / total) for count in counts.values())
        score += entropy * (1 + counts.get(guess[pos], 0) / total)
    return score


def hybrid_heuristic(candidates: Sequence[str], guess: str) -> float:
    """Adaptive heuristic used by the optimized entropy-based solver."""
    n = len(candidates)
    if n <= 2:
        return 1000.0 if guess in candidates else 0.0
    if n <= 10:
        return minimax_heuristic(candidates, guess) + (0.5 if guess in candidates else 0.0)
    return fast_entropy_heuristic(candidates, guess)


_pattern_cache: Dict[Tuple[str, Tuple[str, ...]], float] = {}


def cached_pattern_heuristic(candidates: Sequence[str], guess: str) -> float:
    """Memoize expensive pattern scores for repeated batch comparisons."""
    cache_key = (guess, tuple(sorted(candidates[:100])))
    if cache_key not in _pattern_cache:
        _pattern_cache[cache_key] = fast_entropy_heuristic(candidates, guess) if len(candidates) > 150 else minimax_heuristic(candidates, guess)
    return _pattern_cache[cache_key]


HEURISTICS: Dict[str, Callable[[Sequence[str], str], float]] = {
    "fast_entropy": fast_entropy_heuristic,
    "minimax": minimax_heuristic,
    "expected_size": expected_size_heuristic,
    "hybrid": hybrid_heuristic,
    "positional": positional_entropy_heuristic,
    "cached": cached_pattern_heuristic,
}


def astar_next_guess(candidates: Sequence[str], history: History, opening: str = "RAISE", heuristic: str = "hybrid") -> str:
    """Choose the next optimized entropy/A* guess from the candidate set."""
    candidates = list(candidates)
    if not candidates:
        return random_guess(candidates, history)

    opening = normalize_word(opening)
    if not history:
        return opening if opening in WORD_SET else "RAISE"
    if len(candidates) <= 2:
        return candidates[0]

    if len(candidates) > 100:
        frequencies = Counter(letter for word in candidates for letter in set(word))
        test_words = sorted(candidates, key=lambda word: sum(frequencies[letter] for letter in set(word)), reverse=True)[:50]
    else:
        test_words = candidates

    score_guess = HEURISTICS.get(heuristic, hybrid_heuristic)
    return max(test_words, key=lambda guess: score_guess(candidates, guess))


def entropy_based_guess(candidates: Sequence[str], history: History | None = None) -> str:
    """Portfolio-facing name for the optimized entropy-based AI."""
    return astar_next_guess(candidates, history or [], heuristic="hybrid")


# Backwards-compatible names used by the original notebook UI.
ai_guess_cbp_fixed = set_frequency_guess


AI_STRATEGIES: Dict[str, Callable[[Sequence[str], History], str]] = {
    "Random": random_guess,
    "Random Candidate": random_candidate_guess,
    "Positional Frequency": positional_frequency_guess,
    "Set Frequency": set_frequency_guess,
    "Entropy-based AI": entropy_based_guess,
}


## Game Modes

In [5]:
def solve_secret(secret_word: str, strategy: Callable[[Sequence[str], History], str], max_attempts: int = MAX_ATTEMPTS) -> History:
    """Run one AI strategy against a secret word and return the guess history."""
    secret_word = normalize_word(secret_word)
    candidates = WORDS.copy()
    history: History = []

    for _ in range(max_attempts):
        guess = normalize_word(strategy(candidates, history))
        feedback = get_feedback(guess, secret_word)
        history.append((guess, feedback))
        if guess == secret_word:
            break
        candidates = filter_words(candidates, guess, feedback)

    return history


def new_game_cbp():
    """Start a player-vs-set-frequency-AI game."""
    state = {
        "secret": random.choice(WORDS),
        "player_hist": [],
        "ai_hist": [],
        "candidates": WORDS.copy(),
        "turn": "Player",
        "finished": False,
    }
    return render_grid([]), render_grid([]), "New Set Frequency game started. Secret chosen.", state


def play_turn_cbp(player_guess: str, state: dict):
    """Play one player turn followed by one set-frequency AI turn."""
    return play_turn_with_ai(player_guess, state, ai_guess_cbp_fixed, "Set Frequency AI")


def new_game_astar(opening: str, heuristic: str):
    """Start a player-vs-entropy/A* game."""
    state = {
        "secret": random.choice(WORDS),
        "player_hist": [],
        "ai_hist": [],
        "candidates": WORDS.copy(),
        "opening": normalize_word(opening),
        "heuristic": heuristic,
        "turn": "Player",
        "finished": False,
    }
    return render_grid([]), render_grid([]), f"New Entropy-based game started with the {heuristic} heuristic.", state


def play_turn_astar(player_guess: str, state: dict):
    """Play one player turn followed by one entropy/A* AI turn."""
    if state is None:
        return render_grid([]), render_grid([]), "Start a new Entropy-based game first.", None

    def strategy(candidates: Sequence[str], history: History) -> str:
        return astar_next_guess(candidates, history, opening=state.get("opening", "RAISE"), heuristic=state.get("heuristic", "hybrid"))

    return play_turn_with_ai(player_guess, state, strategy, "Entropy-based AI")


def play_turn_with_ai(player_guess: str, state: dict, ai_strategy: Callable[[Sequence[str], History], str], ai_name: str):
    """Shared player-vs-AI turn handler used by both interactive modes."""
    if state is None:
        return render_grid([]), render_grid([]), "Start a new game first.", None
    if state["finished"]:
        return render_grid(state["player_hist"]), render_grid(state["ai_hist"]), "Game finished.", state

    guess = normalize_word(player_guess)
    if not is_valid_guess(guess):
        return render_grid(state["player_hist"]), render_grid(state["ai_hist"]), "Invalid guess. Enter a word from the dataset.", state

    feedback = get_feedback(guess, state["secret"])
    state["player_hist"].append((guess, feedback))
    if guess == state["secret"]:
        state["finished"] = True
        return render_grid(state["player_hist"]), render_grid(state["ai_hist"]), "Player wins!", state

    ai_guess = ai_strategy(state["candidates"], state["ai_hist"])
    ai_feedback = get_feedback(ai_guess, state["secret"])
    state["ai_hist"].append((ai_guess, ai_feedback))
    state["candidates"] = filter_words(state["candidates"], ai_guess, ai_feedback)

    if ai_guess == state["secret"]:
        state["finished"] = True
        return render_grid(state["player_hist"]), render_grid(state["ai_hist"]), f"{ai_name} wins. The word was {state['secret']}.", state

    if len(state["player_hist"]) >= MAX_ATTEMPTS or len(state["ai_hist"]) >= MAX_ATTEMPTS:
        state["finished"] = True
        return render_grid(state["player_hist"]), render_grid(state["ai_hist"]), f"Out of turns. The word was {state['secret']}.", state

    return render_grid(state["player_hist"]), render_grid(state["ai_hist"]), "Turn: Player", state


def ai_vs_ai_solve(secret_word: str, heuristic_choice: str):
    """Compare the entropy/A* solver with the set-frequency solver on one word."""
    secret_word = normalize_word(secret_word)
    if not is_valid_guess(secret_word):
        return f"Invalid secret word '{secret_word}'.", render_grid([]), render_grid([])

    astar_history = solve_secret(
        secret_word,
        lambda candidates, history: astar_next_guess(candidates, history, heuristic=heuristic_choice),
    )
    frequency_history = solve_secret(secret_word, set_frequency_guess)

    astar_solved = astar_history[-1][0] == secret_word
    frequency_solved = frequency_history[-1][0] == secret_word
    astar_count = len(astar_history)
    frequency_count = len(frequency_history)

    if astar_solved and frequency_solved:
        if astar_count < frequency_count:
            summary = f"Entropy-based AI wins in {astar_count} attempts vs {frequency_count}."
        elif frequency_count < astar_count:
            summary = f"Set Frequency AI wins in {frequency_count} attempts vs {astar_count}."
        else:
            summary = f"Both AIs solved it in {astar_count} attempts."
    elif astar_solved:
        summary = f"Entropy-based AI solved it in {astar_count}; Set Frequency AI failed."
    elif frequency_solved:
        summary = f"Set Frequency AI solved it in {frequency_count}; Entropy-based AI failed."
    else:
        summary = f"Both AIs failed to solve {secret_word} within {MAX_ATTEMPTS} attempts."

    return summary, render_grid(astar_history), render_grid(frequency_history)


## Gradio Interface

In [6]:
@dataclass
class BenchmarkResult:
    strategy: str
    games: int
    successes: int
    average_guesses: float
    success_rate: float
    average_runtime_ms: float


def benchmark_strategy(name: str, strategy: Callable[[Sequence[str], History], str], secrets: Sequence[str]) -> BenchmarkResult:
    """Benchmark one AI strategy over a fixed list of secret words."""
    guess_counts: List[int] = []
    successes = 0
    start = time.perf_counter()

    for secret in secrets:
        history = solve_secret(secret, strategy)
        solved = bool(history and history[-1][0] == secret)
        if solved:
            successes += 1
            guess_counts.append(len(history))

    elapsed = time.perf_counter() - start
    games = len(secrets)
    return BenchmarkResult(
        strategy=name,
        games=games,
        successes=successes,
        average_guesses=(sum(guess_counts) / len(guess_counts)) if guess_counts else float("nan"),
        success_rate=successes / games if games else 0.0,
        average_runtime_ms=(elapsed / games * 1000) if games else 0.0,
    )


def run_ai_benchmark(num_games: int = 300, seed: int = 42):
    """Compare all five AI difficulty levels over hundreds of random games."""
    rng = random.Random(seed)
    secrets = [rng.choice(WORDS) for _ in range(int(num_games))]
    results = [benchmark_strategy(name, strategy, secrets) for name, strategy in AI_STRATEGIES.items()]

    rows = [result.__dict__ for result in results]
    if pd is not None:
        table = pd.DataFrame(rows)
        table["success_rate"] = (table["success_rate"] * 100).round(2)
        table["average_guesses"] = table["average_guesses"].round(2)
        table["average_runtime_ms"] = table["average_runtime_ms"].round(3)
        return table.rename(columns={
            "strategy": "AI Strategy",
            "games": "Games",
            "successes": "Successes",
            "average_guesses": "Average Guesses",
            "success_rate": "Success Rate (%)",
            "average_runtime_ms": "Average Runtime (ms)",
        })

    return rows


def benchmark_markdown(num_games: int = 300, seed: int = 42) -> str:
    """Return benchmark results as a Markdown table for the Gradio interface."""
    results = run_ai_benchmark(num_games=num_games, seed=seed)
    rows = results.to_dict("records") if pd is not None else results
    headers = list(rows[0].keys()) if rows else []
    table = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        table.append("| " + " | ".join(str(row[header]) for header in headers) + " |")
    return "\n".join(table)

with gr.Blocks(title="AI Wordle Solver", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# AI Wordle Solver")
    gr.Markdown("A single-notebook portfolio project comparing five Wordle AI strategies.")

    with gr.Tab("Set Frequency vs Player"):
        with gr.Row():
            with gr.Column(scale=1):
                start_btn_cbp = gr.Button("Start New Game", variant="primary")
                guess_in_cbp = gr.Textbox(label="Your Guess", placeholder="RAISE")
                play_btn_cbp = gr.Button("Play Turn")
                status_cbp = gr.Textbox(label="Game Status", interactive=False)
            with gr.Column(scale=2):
                gr.Markdown("### Player Board")
                player_out_cbp = gr.HTML(render_grid([]))
            with gr.Column(scale=2):
                gr.Markdown("### AI Board")
                ai_out_cbp = gr.HTML(render_grid([]))

        state_cbp = gr.State()
        start_btn_cbp.click(new_game_cbp, outputs=[player_out_cbp, ai_out_cbp, status_cbp, state_cbp])
        play_btn_cbp.click(play_turn_cbp, inputs=[guess_in_cbp, state_cbp], outputs=[player_out_cbp, ai_out_cbp, status_cbp, state_cbp])

    with gr.Tab("Entropy AI vs Player"):
        with gr.Row():
            with gr.Column(scale=1):
                opening_choice_astar = gr.Textbox(label="AI Opening Guess", value="RAISE")
                heuristic_choice_astar = gr.Dropdown(
                    label="Heuristic",
                    choices=list(HEURISTICS.keys()),
                    value="hybrid",
                )
                start_btn_astar = gr.Button("Start New Game", variant="primary")
                guess_in_astar = gr.Textbox(label="Your Guess", placeholder="RAISE")
                play_btn_astar = gr.Button("Play Turn")
                status_astar = gr.Textbox(label="Game Status", interactive=False)
            with gr.Column(scale=2):
                gr.Markdown("### Player Board")
                player_out_astar = gr.HTML(render_grid([]))
            with gr.Column(scale=2):
                gr.Markdown("### AI Board")
                ai_out_astar = gr.HTML(render_grid([]))

        state_astar = gr.State()
        start_btn_astar.click(new_game_astar, inputs=[opening_choice_astar, heuristic_choice_astar], outputs=[player_out_astar, ai_out_astar, status_astar, state_astar])
        play_btn_astar.click(play_turn_astar, inputs=[guess_in_astar, state_astar], outputs=[player_out_astar, ai_out_astar, status_astar, state_astar])

    with gr.Tab("AI vs AI"):
        with gr.Row():
            secret_for_comparison = gr.Textbox(label="Secret Word", placeholder="WORLD")
            heuristic_for_comparison = gr.Dropdown(label="Entropy Heuristic", choices=list(HEURISTICS.keys()), value="hybrid")
            compare_btn = gr.Button("Run Comparison", variant="primary")

        comparison_summary = gr.Textbox(label="Result", interactive=False, lines=3)
        with gr.Row():
            with gr.Column():
                gr.Markdown("### Entropy-based AI")
                astar_out_compare = gr.HTML(render_grid([]))
            with gr.Column():
                gr.Markdown("### Set Frequency AI")
                cbp_out_compare = gr.HTML(render_grid([]))

        compare_btn.click(ai_vs_ai_solve, inputs=[secret_for_comparison, heuristic_for_comparison], outputs=[comparison_summary, astar_out_compare, cbp_out_compare])

    with gr.Tab("Benchmarking"):
        gr.Markdown("## Five-strategy benchmark")
        with gr.Row():
            benchmark_games = gr.Slider(minimum=100, maximum=1000, step=50, value=300, label="Random Games")
            benchmark_seed = gr.Number(value=42, precision=0, label="Random Seed")
            benchmark_btn = gr.Button("Run Benchmark", variant="primary")
        benchmark_output = gr.Markdown(benchmark_markdown(300, 42))
        benchmark_btn.click(lambda games, seed: benchmark_markdown(int(games), int(seed)), inputs=[benchmark_games, benchmark_seed], outputs=[benchmark_output])

    with gr.Tab("Strategy Guide"):
        gr.Markdown("""
## AI Difficulty Levels

| Strategy | Behavior |
| --- | --- |
| Random | Guesses any word from the full dataset. |
| Random Candidate | Guesses randomly from words still consistent with feedback. |
| Positional Frequency | Prefers candidates with common letters in common positions. |
| Set Frequency | Preserves the original constraint-based unique-letter frequency solver. |
| Entropy-based AI | Preserves the optimized entropy/A* solver with adaptive heuristics. |

The benchmark tab compares all five strategies using the same randomly sampled secret words.
        """)

# Launch the app from the notebook.
demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Benchmarking

The benchmarking helpers below compare all five AI difficulty levels across hundreds of random games and report average guesses, success rate, and average runtime.

In [7]:
# Run this cell to generate the portfolio benchmark report.
benchmark_report = run_ai_benchmark(num_games=300, seed=42)
benchmark_report


,AI Strategy,Games,Successes,Average Guesses,Success Rate (%),Average Runtime (ms)
0,Random,300,4,3.00,1.33,0.682
1,Random Candidate,300,299,3.38,99.67,0.691
2,Positional Frequency,300,300,3.06,100.00,0.919
3,Set Frequency,300,300,2.92,100.00,0.993
4,Entropy-based AI,300,300,3.00,100.00,0.959
